In [ ]:
import pandas as pd
import gc
import missingno as msno
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib

from sklearn.model_selection import train_test_split, cross_validate, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
hitc = pd.read_parquet("Model/ga_hits-001.parquet")

In [ ]:
hits = hitc[["session_id", "event_action"]].copy()


In [ ]:
del hitc

In [ ]:
gc.collect()

In [ ]:
target_actions = ['sub_car_claim_click', 'sub_car_claim_submit_click', 'sub_open_dialog_click', 'sub_custom_question_submit_click', 'sub_call_number_click', 'sub_callback_submit_click', 'sub_submit_success', 'sub_car_request_submit_click']

In [ ]:
hits['is_target'] = hits['event_action'].isin(target_actions).astype('int8')

In [ ]:
target_df = (hits.groupby('session_id', as_index=False)['is_target'].max().rename(columns={'is_target': 'target'}))

In [ ]:
target_df.target.value_counts(normalize=True)

In [ ]:
del hits

In [ ]:
gc.collect()

In [ ]:
sessions = pd.read_parquet("Model/ga_sessions.parquet")

In [ ]:
sessions['session_id'] = sessions['session_id'].astype(str)

In [ ]:
target_df['session_id'] = target_df['session_id'].astype(str)

In [ ]:
df = sessions.merge(target_df, on='session_id', how='left')

In [ ]:
df["target"] = df["target"].fillna(0).astype("int8")

In [ ]:
df.head()

## Data preparation

### Data cleaning

In [ ]:
print('Sample size: {}, {}'.format(df.shape[0], df.shape[1]))

In [ ]:
print('Sample info:\n')
df.info()

#### Duplicates cleaning

In [ ]:
print('Duplicate check:')
df.duplicated(subset="session_id").sum()

#### Missing Values Handling

In [ ]:
print('Features with the most missing values')
msno.matrix(df)

In [ ]:
missing_values = ((df.isna().sum() / len(df)) * 100).sort_values(ascending=False)
print('Percentage of missing values:')
missing_values

In [ ]:
df = df.drop(columns=["device_model"])

In [ ]:
df["utm_source"] = df["utm_source"].fillna("unknown_source")

In [ ]:
print(df.groupby("device_category")["device_os"].apply(lambda x: x.isna().mean() * 100))

In [ ]:
print(df[df["device_os"].isna()]["device_browser"].value_counts().head(15))

In [ ]:
df["device_os"] = df["device_os"].fillna("os_not_detected")

In [ ]:
# for these three columns the hypothesis is the same: missing values are most likely tied to organic/direct traffic, which physically can't have ad attributes (no campaign -> no keyword -> no ad content)
for col in ["utm_keyword", "utm_campaign", "utm_adcontent"]:
    print(f"--- {col} ---")
    print(df.groupby("utm_medium")[col].apply(lambda x: x.isna().mean() * 100))
    print()

In [ ]:
# The hypothesis worked almost perfectly here — missing values are concentrated in cpc (48.7%) and referral (3.7%), 0% for most other types
df["utm_campaign"] = df["utm_campaign"].fillna("no_campaign")

In [ ]:
# Similar picture — missing values in cpa (88.5%), cpc (73.9%), the rest close to zero
df["utm_adcontent"] = df["utm_adcontent"].fillna("no_adcontent")

In [ ]:
# The specific keyword value is unreliable and noisy, but the mere fact of "present or not" can be a weak but meaningful signal — better to reduce it to a simple flag than feed the model near-random noise from hundreds of unique words
df["has_keyword"] = df["utm_keyword"].notna().astype("int8")
df = df.drop(columns=["utm_keyword"])

In [ ]:
# Check the share of missing brands by device
print(df.groupby("device_category")["device_brand"].apply(lambda x: x.isna().mean() * 100))

In [ ]:
print(f'Missing count for mobile: {df[(df["device_category"] == "mobile") & (df["device_brand"].isna())].shape[0]}')
print(f'Missing count for tablet: {df[(df["device_category"] == "tablet") & (df["device_brand"].isna())].shape[0]}')

In [ ]:
print(df[(df["device_category"] == "mobile") & (df["device_brand"].isna())]["device_os"].value_counts())
print(df[(df["device_category"] == "tablet") & (df["device_brand"].isna())]["device_os"].value_counts())

In [ ]:
# Filling in missing values
mask_desktop = df["device_category"] == "desktop"
mask_other_missing = df["device_brand"].isna() & ~mask_desktop
df.loc[mask_desktop, "device_brand"] = df.loc[mask_desktop, "device_brand"].fillna("desktop_no_brand")
df.loc[mask_other_missing, "device_brand"] = "brand_unknown"

In [ ]:
missing_values = ((df.isna().sum() / len(df)) * 100).sort_values(ascending=False)
print('Percentage of missing values:')
missing_values

#### Data Types Correction

In [ ]:
df.dtypes

In [ ]:
df["visit_date"] = pd.to_datetime(df["visit_date"])

In [ ]:
df["visit_number"] = df["visit_number"].astype("int32")

In [ ]:
categorical_cols = [
    "utm_source", "utm_medium", "utm_campaign", "utm_adcontent",
    "device_category", "device_os", "device_brand",
    "device_browser", "geo_country", "geo_city"]

for col in categorical_cols:
    df[col] = df[col].astype("category")

In [ ]:
print(df.dtypes)
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")

#### Outliers and Anomalies Handling

In [ ]:
df_clean = df.copy()

In [ ]:
def calculate_outliers(data):
    q25 = data.quantile(0.25)
    q75 = data.quantile(0.75)
    iqr = q75 - q25
    boundaries = (q25 - 1.5 * iqr, q75 + 1.5 * iqr)
    return boundaries

##### Outlier and anomaly analysis for visit_number

In [ ]:
df_clean.visit_number.describe()

In [ ]:
# How target is distributed depending on the number of visits
print(df_clean.groupby("visit_number")["target"].agg(["count", "mean"]).head(20))

In [ ]:
q99 = df_clean["visit_number"].quantile(0.99)
print(f"99th percentile: {q99}")

print("CR for regular users (<=99th percentile):")
print(df_clean[df_clean["visit_number"] <= q99]["target"].mean())

print("CR for extreme users (>99th percentile):")
print(df_clean[df_clean["visit_number"] > q99]["target"].mean())

print("Row count in each group:")
print((df_clean["visit_number"] <= q99).sum())
print((df_clean["visit_number"] > q99).sum())

In [ ]:
q999 = df_clean["visit_number"].quantile(0.999)
print(f"99.9th percentile: {q999}")

print("CR for users above the 99.9th percentile:")
print(df_clean[df_clean["visit_number"] > q999]["target"].mean())
print("Number of such rows:")
print((df_clean["visit_number"] > q999).sum())

Checking visit_number showed that CR grows with the number of visits: 2.69% for users at or below the 99th percentile vs 4.03% above the 99th and 11.34% above the 99.9th percentile. The extreme values turned out to be a strong predictor of conversion rather than an anomaly, so the column was left unchanged.

##### Outlier and anomaly analysis for visit_date

In [ ]:
print(df_clean["visit_date"].min())
print(df_clean["visit_date"].max())

In [ ]:
daily_counts = df_clean["visit_date"].value_counts().sort_index()
print(daily_counts.describe())

In [ ]:
print("Days with the fewest visits:")
print(daily_counts.sort_values().head(10))

print("Days with the most visits:")
print(daily_counts.sort_values(ascending=False).head(10))

The distribution of visits by date showed three notable deviations from the average (8,230 visits/day): a drop in late May 2021 (877–1,246 visits) right before the service launch, a sharp spike on May 24, 2021 (39,453 visits) — the launch day — and a peak on December 21, 2021 (30,704 visits), likely tied to pre-New-Year activity. All deviations are explained by real business events rather than data errors, so the visit_date column was left unchanged.

##### Outlier and anomaly analysis for device_screen_resolution

In [ ]:
print(df_clean["device_screen_resolution"].value_counts().head(15))

In [ ]:
print(df_clean["device_screen_resolution"].value_counts().tail(15))

In [ ]:
print(df_clean["device_screen_resolution"].nunique())

In [ ]:
suspicious = df_clean["device_screen_resolution"].isin(["0x0", "(not set)", "1x1", ""])
print(suspicious.sum())

In [ ]:
print(df_clean[suspicious]["device_screen_resolution"].value_counts())

In [ ]:
df_clean.loc[suspicious, "device_screen_resolution"] = np.nan

In [ ]:
print(df_clean["device_screen_resolution"].isna().sum())

Checking device_screen_resolution for technical anomalies revealed 19 rows (0.001% of the dataset) with placeholder values "0x0" (11 rows) and "(not set)" (8 rows). These values were converted to missing and will later be filled with median values when the feature is split into screen_width and screen_height, given their negligible volume.

#### Final Data Cleaning Check

After handling missing values, duplicates, data types, and anomalies, a final quality check of the cleaned dataset was performed.

In [ ]:
print(df_clean.shape)
print(df_clean.memory_usage(deep=True).sum() / 1024**2, "MB")

In [ ]:
print(df_clean.isna().sum())

In [ ]:
df_clean.duplicated(subset="session_id").sum()

In [ ]:
df_clean.dtypes

In [ ]:
df_clean.head()

In [ ]:
df_clean.describe(include="all").T

In [ ]:
df_clean["utm_adcontent"].value_counts().head(5)

In [ ]:
df_clean[df_clean["utm_adcontent"] == "JNHcPlZPxEMWDnRiyoBf"]["utm_medium"].value_counts()

Checking the dominant value of utm_adcontent ("JNHcPlZPxEMWDnRiyoBf", 54% of the dataset) revealed that it appears across different traffic types (banner, (none), referral, cpc, organic, etc.), which points to a technical labeling quirk (likely a default/generic tag) rather than a specific ad creative. No changes required.

Everything looks correctly finalized, except for the intentional 19 NaNs in device_screen_resolution, which are deliberately left for handling at the feature engineering stage. No duplicates, correct types, missing values resolved (except the deferred ones).

### Feature Selection

In [ ]:
df_model = df_clean.copy()

In [ ]:
for col in ["utm_source", "utm_medium", "utm_campaign", "utm_adcontent", 
            "device_category", "device_os", "device_brand", "device_browser",
            "geo_country", "geo_city"]:
    top_share = df_model[col].value_counts(normalize=True).iloc[0]
    print(f"{col}: {top_share:.4f} ({df_clean[col].nunique()} unique)")

Categorical feature analysis showed that most columns have an acceptable distribution (top-value share 25-58%), except geo_country, where a single value (Russia) accounts for 96.8% of the data across 166 unique countries — a clear candidate for simplification into a binary feature. Columns utm_source (294), utm_campaign (413), device_brand (208) and geo_city (2548) also have high cardinality (many unique values) with a relatively low top-value share — i.e. the data is spread thin across many categories.

#### Geo_country FS

In [ ]:
df_model["is_russia"] = (df_model["geo_country"] == "Russia").astype("int8")

In [ ]:
print(df_model.groupby("is_russia")["target"].agg(["count", "mean"]))

Checking the relationship between geo_country and the target showed that Russian traffic converts more often (CR = 2.73%) compared to other countries (CR = 1.95%, n=59,477). The difference is moderate but the feature retains a useful signal. Decision made to replace the geo_country column (166 unique values) with a binary is_russia feature, since this simplification preserves the main pattern and removes the high-cardinality problem caused by a long tail of rare countries.

In [ ]:
df_model = df_model.drop(columns=["geo_country"])

#### Geo_city FS

In [ ]:
df_model["geo_city"].value_counts().head(10)

In [ ]:
df_model["is_presence_city"] = df_model["geo_city"].isin(["Moscow", "Saint Petersburg"]).astype("int8")
df_model.groupby("is_presence_city")["target"].agg(["count", "mean"])

In [ ]:
for n in [10, 15, 20, 25, 30]:
    top_n_cities = df_model["geo_city"].value_counts().head(n).index
    coverage = df_model["geo_city"].isin(top_n_cities).mean()
    print(f"Top-{n} cities cover {coverage*100:.1f}% of all visits")

Grouping geo_city: choosing the top-N size

The 'geo_city' column contains 2548 unique cities, most of which 
appear only once — this is high cardinality, 
unsuitable for direct use in the model

To choose the optimal grouping threshold, data coverage was checked 
at different top sizes:\
Top-10 cities cover 73.6% of all visits\
Top-15 cities cover 77.6% of all visits\
Top-20 cities cover 80.9% of all visits\
Top-25 cities cover 83.0% of all visits\
Top-30 cities cover 84.4% of all visits

The coverage gain gradually tapers off, indicating diminishing returns from adding more cities.
Top-20 was chosen as the balance between data coverage (80.9%) and the number of resulting categories. The remaining cities will be merged into the "other" category

In [ ]:
top_20_cities = df_model["geo_city"].value_counts().head(20).index
df_model["geo_city_grouped"] = np.where(df_model["geo_city"].astype(str).isin(top_20_cities), df_model["geo_city"].astype(str), "other")
df_model["geo_city_grouped"] = df_model["geo_city_grouped"].astype("category")
print(df_model["geo_city_grouped"].value_counts(), df_model["geo_city_grouped"].dtype)

The geo_city column (2548 unique values) was grouped into geo_city_grouped: the top 20 cities by frequency were kept (covering 80.9% of visits), the rest merged into an "other" category (354,709 rows, 19.1%). This reduces the feature's cardinality from 2548 to 21 categories while preserving most of the information

In [ ]:
df_model = df_model.drop(columns=["geo_city"])

#### Utm_source FS

In [ ]:
df_model["utm_source"].value_counts().head(15)

In [ ]:
for n in [10, 15, 20, 25, 30]:
    top_n = df_model["utm_source"].value_counts().head(n).index
    coverage = df_clean["utm_source"].isin(top_n).mean()
    print(f"Top-{n} sources cover {coverage*100:.1f}% of all visits")

Grouping utm_source: choosing the top-N size

The 'utm_source' column contains 294 unique values. Checking data 
coverage at different top sizes showed:\
Top-10 sources cover 90.9% of all visits\
Top-15 sources cover 95.3% of all visits\
Top-20 sources cover 97.3% of all visits

Already top-10 provides high coverage (90.9%), and further increases 
yield only marginal gains. Top-10 was chosen as the optimal balance 
between data coverage and feature compactness

In [ ]:
top_10_utm_sources = df_model["utm_source"].value_counts().head(10).index
df_model["utm_source_grouped"] = np.where(df_model["utm_source"].astype(str).isin(top_10_utm_sources), df_model["utm_source"].astype(str), "other")
df_model["utm_source_grouped"] = df_model["utm_source_grouped"].astype("category")
print(df_model["utm_source_grouped"].value_counts(), df_model["utm_source_grouped"].dtype)

The `utm_source` column was grouped into `utm_source_grouped`: the 
top 10 sources by frequency were kept, the rest merged into the "other" category 
(170,154 rows, 9.1%). The feature's cardinality was reduced from 294 to 11 categories.

In [ ]:
df_model = df_model.drop(columns=["utm_source"])

#### Utm_compaign FS

In [ ]:
df_model["utm_campaign"].value_counts().head(10)

In [ ]:
for n in [10, 15, 20, 25, 30]:
    top_n = df_model["utm_campaign"].value_counts().head(n).index
    coverage = df_model["utm_campaign"].isin(top_n).mean()
    print(f"Top-{n} ad campaigns cover {coverage*100:.1f}% of all visits")

Grouping utm_campaign: choosing the top-N size

The utm_campaign column contains 294 unique values. Checking data 
coverage at different top sizes showed:\
Top-10 campaigns cover 81.0% of all visits\
Top-15 campaigns cover 85.0% of all visits\
Top-20 campaigns cover 87.7% of all visits\
Top-25 campaigns cover 89.7% of all visits\
Top-30 campaigns cover 91.5% of all visits

Top-15 provides high coverage (85%), and further increases 
yield only marginal gains. Top-15 was chosen as the optimal balance 
between data coverage and feature compactness (~85%). The remaining sources 
were merged into the "other" category.

In [ ]:
top_15_utm_campaign = df_model["utm_campaign"].value_counts().head(15).index
df_model["utm_campaign_grouped"] = np.where(df_model["utm_campaign"].astype(str).isin(top_15_utm_campaign), df_model["utm_campaign"].astype(str), "other")
df_model["utm_campaign_grouped"] = df_model["utm_campaign_grouped"].astype("category")
print(df_model["utm_campaign_grouped"].value_counts(), df_model["utm_campaign_grouped"].dtype)

The utm_campaign column (413 unique values) was grouped into utm_campaign_grouped: the top 15 campaigns by frequency were kept (covering 85.0% of visits), the rest merged into "other" (279,513 rows, 15.0%). The feature's cardinality was reduced from 413 to 16 categories

In [ ]:
df_model = df_model.drop(columns="utm_campaign")

#### Device_brand FS

In [ ]:
df_model["device_brand"].value_counts().head(10)

In [ ]:
print(repr(df_model["device_brand"].value_counts().index[3]))

In [ ]:
df_model["device_brand"] = df_model["device_brand"].astype(str)
df_model["device_brand"] = df_model["device_brand"].replace('', np.nan)
print(df_model["device_brand"].isna().sum())

In [ ]:
df_model[df_model["device_brand"].isna()]["device_category"].value_counts()

In [ ]:
mask_desktop = df_model["device_category"] == "desktop"
mask_other_missing = df_model["device_brand"].isna() & ~mask_desktop
df_model.loc[mask_desktop, "device_brand"] = df_model.loc[mask_desktop, "device_brand"].fillna("desktop_no_brand")
df_model.loc[mask_other_missing, "device_brand"] = "brand_unknown"

In [ ]:
df_model["device_brand"] = df_model["device_brand"].astype(str)
print(df_model["device_brand"].isna().sum())

In [ ]:
df_model[df_model["device_brand"]=="(not set)"]["device_category"].value_counts()

In [ ]:
mask_desktop_ns = (df_model["device_category"] == "desktop") & (df_model["device_brand"] == "(not set)")
mask_other_ns = (df_model["device_category"] != "desktop") & (df_model["device_brand"] == "(not set)")
df_model.loc[mask_desktop_ns, "device_brand"] = "desktop_no_brand"
df_model.loc[mask_other_ns, "device_brand"] = "brand_unknown"

In [ ]:
df_model["device_brand"] = df_model["device_brand"].astype("category")

In [ ]:
df_model["device_brand"].value_counts().head(15)

Checking device_brand revealed two types of irregular values:
248,500 rows with an empty string ' ' (not recognized as NaN) — after converting to NaN, replaced with desktop_no_brand (for desktop) and brand_unknown (for mobile/tablet);
17,545 rows with the value "(not set)" (mostly mobile and tablet) — merged into brand_unknown (mobile/tablet) and desktop_no_brand (a handful of desktop)

In [ ]:
for n in [5, 10, 15, 20]:
    top_n = df_model["device_brand"].value_counts().head(n).index
    coverage = df_model["device_brand"].isin(top_n).mean()
    print(f"Top-{n} brands cover {coverage*100:.1f}% of all visits")

Grouping device_brand: choosing the top-N size.
After handling missing values and placeholder tags, the device_brand column 
showed high concentration — the top 5 brands cover 92.7% of visits. 
Checking at different top sizes:
Top-5 brands cover 92.7% of all visits\
Top-10 brands cover 96.6% of all visits\
Top-15 brands cover 98.4% of all visits\
Top-20 brands cover 99.2% of all visits

Top-10 (96.6% coverage) was chosen as the balance between data completeness and 
feature compactness. The remaining brands were merged into the "other" category.

In [ ]:
top_5_device_brand = df_model["device_brand"].value_counts().head(10).index
df_model["device_brand_grouped"] = np.where(df_model["device_brand"].astype(str).isin(top_5_device_brand), df_model["device_brand"].astype(str), "other")
df_model["device_brand_grouped"] = df_model["device_brand_grouped"].astype("category")
print(df_model["device_brand_grouped"].value_counts(), df_model["device_brand_grouped"].dtype)

The device_brand column (208 unique values) was grouped into 
device_brand_grouped: the top 10 brands by frequency were kept (covering 
96.6% of visits), the rest merged into "other" (63,550 rows, 
3.4%). The feature's cardinality was reduced from 208 to 11 categories.

In [ ]:
df_model = df_model.drop(columns="device_brand")

#### Utm_adcontent FS

In [ ]:
df_model["utm_adcontent"].value_counts().head(10)

In [ ]:
for n in [10, 15, 20, 25, 30]:
    top_n = df_model["utm_adcontent"].value_counts().head(n).index
    coverage = df_model["utm_adcontent"].isin(top_n).mean()
    print(f"Top-{n} ad content values cover {coverage*100:.1f}% of all visits")

Grouping utm_adcontent: choosing the top-N size

The utm_adcontent column contains 287 unique values. Checking 
data coverage at different top sizes showed:\
Top-10 values cover 95.3% of all visits\
Top-15 values cover 97.1% of all visits\
Top-20 values cover 98.0% of all visits

Already top-10 provides high coverage (95.3%), further increases 
yield marginal gains. Top-10 was chosen as 
the optimal balance between data coverage and feature compactness.

In [ ]:
top_10_utm_adcontent = df_model["utm_adcontent"].value_counts().head(10).index
df_model["utm_adcontent_grouped"] = np.where(df_model["utm_adcontent"].astype(str).isin(top_10_utm_adcontent), df_model["utm_adcontent"].astype(str), "other")
df_model["utm_adcontent_grouped"] = df_model["utm_adcontent_grouped"].astype("category")
print(df_model["utm_adcontent_grouped"].value_counts(), df_model["utm_adcontent_grouped"].dtype)

In [ ]:
df_model = df_model.drop(columns= "utm_adcontent")

In [ ]:
print(df_model.columns.tolist())

In [ ]:
df_model.isna().sum()

In [ ]:
df_model.dtypes

In [ ]:
df_model.shape

Feature Selection stage summary:
    Session_id and client_id were kept in the dataframe (useful as identifiers for matching predictions) but excluded from the model's feature list — they are unique per row and carry no generalizable information.
    Before processing features, their relationship with the target was checked, which helped avoid dropping valuable predictors (for example, visit_number formally met the outlier criteria but turned out to be one of the strongest features).
    Cardinality of high-cardinality features was reduced via top-N + "other" grouping: geo_country (166 to 2), geo_city (2548 to 21), utm_source (294 to 11), utm_campaign (413 to 16), device_brand (208 to 11), utm_adcontent (287 to 11).
    Additionally, hidden missing values were found and fixed: empty strings and placeholders "(not set)"/"0x0" in device_brand and device_screen_resolution.

Result: 1,860,042 rows, 20 columns. No missing values except 
19 rows in device_screen_resolution (to be handled in the next stage).

### Feature Engineering

#### Visit_weekday FE

In [ ]:
df_model["visit_weekday"] = df_model["visit_date"].dt.dayofweek

In [ ]:
print(df_model["visit_weekday"].value_counts().sort_index())

In [ ]:
print(df_model.groupby("visit_weekday")["target"].agg(["count", "mean"]))

Checking the relationship between visit_weekday and the target showed CR decreasing from the start of the week toward the weekend: from 3.15% on Monday to ~2.45% Friday-Sunday. The difference is moderate but consistent, confirming the feature's usefulness for the model.

#### Visit_hour FE

In [ ]:
df_model["visit_hour"] = pd.to_datetime(df_model["visit_time"], format="%H:%M:%S", errors="coerce").dt.hour

In [ ]:
print(df_model["visit_hour"].isna().sum())
print(df_model["visit_weekday"].value_counts().sort_index())

In [ ]:
print(df_model.groupby("visit_hour")["target"].agg(["count", "mean"]))

The visit hour (visit_hour, range 0-23) was extracted from visit_time. 
Checking its relationship with the target showed a clear daily pattern: 
the minimum CR is observed early in the morning (6:00, CR=1.98%), peaking during 
daytime hours (13:00, CR=3.15%), after which conversion gradually declines toward night. 
The difference between the peak and the minimum is about 1.6x, confirming 
visit_hour as a significant predictor

#### Screen_width, Screen_height FE

In [ ]:
df_model[["screen_width", "screen_height"]] = (df_model["device_screen_resolution"].str.split("x", expand=True).astype("float"))

In [ ]:
df_model[["screen_width", "screen_height"]].isna().sum()

In [ ]:
df_model["screen_width"] = df_model["screen_width"].fillna(df_model["screen_width"].median())
df_model["screen_height"] = df_model["screen_height"].fillna(df_model["screen_height"].median())

df_model[["screen_width", "screen_height"]].isna().sum()

In [ ]:
df_model[["screen_width", "screen_height"]].describe()

In [ ]:
print(df_model["screen_height"].quantile([0.95, 0.99, 0.999, 0.9999, 1.0]))
print(df_model["screen_width"].quantile([0.95, 0.99, 0.999, 0.9999, 1.0]))

In [ ]:
print(df_model[df_model["screen_height"] > 3000][["screen_width", "screen_height", "device_category"]].head(10))

In [ ]:
print((df_model["screen_height"] == 20000).sum())
print(df_model[df_model["screen_height"] == 20000]["screen_width"].value_counts())

In [ ]:
print(df_model[df_model["screen_height"] == 20000]["target"].mean())
print(df_model[df_model["screen_height"] != 20000]["target"].mean())

In [ ]:
df_model["screen_height"] = df_model["screen_height"].clip(upper=2160)
df_model["screen_height"].describe()

In [ ]:
df_model["screen_width"].value_counts().tail(10)

The device_screen_resolution column was split into screen_width and 
screen_height. Checking revealed a technical anomaly: 29 rows 
(0.0016%) with a fixed screen_height=20000 at width 1600 
— an unrealistic aspect ratio not found in real 
devices. CR in this group was 0% versus 2.7% in the rest of the data, 
which points to non-human traffic. Values were clipped to 
2160 pixels (the max for modern 4K monitors). Checking 
screen_width did not reveal any anomalies

In [ ]:
df_model = df_model.drop(columns=["visit_time", "device_screen_resolution"])
df_model.columns.tolist()

In [ ]:
df_model.dtypes

In [ ]:
for col in ["visit_weekday", "visit_hour"]:
    stats = df_model.groupby(col)["target"].mean()
    print(f"Feature: {col}, min CR: {stats.min()*100:.2f}%, max CR: {stats.max()*100:.2f}%")

In [ ]:
print(df_model[["screen_width", "screen_height", "target"]].corr()["target"])

Feature Engineering stage summary\
Three new features were created from existing ones:\
visit_weekday - day of week from visit_date (CR decreases from 3.15% on Monday to ~2.45% on weekends)\
visit_hour - hour from visit_time (daytime CR peak at 13:00 - 3.15%, nighttime minimum at 6:00 - 1.98%)\
screen_width, screen_height - from device_screen_resolution (an anomaly was found and clipped: 29 rows with an unrealistic 1600x20000 resolution, likely bots, CR=0%)

visit_weekday and visit_hour showed a noticeable relationship with the target and are promising features for the model.\
screen_width/screen_height did not show a strong linear relationship with the target but were kept in the dataset, as they may reveal more complex, indirect patterns through interaction with other features

All new features showed a relationship with the target and were kept for model training. The original visit_time and device_screen_resolution columns were dropped after the information was extracted

#### Checking feature multicollinearity

In [ ]:
numeric_check_cols = ["visit_number", "has_keyword", "is_russia", "is_presence_city", "visit_weekday", "visit_hour", "screen_width", "screen_height"]

In [ ]:
corr_matrix = df_model[numeric_check_cols].corr()
print(corr_matrix)

A correlation matrix of numeric features was built. Most pairs showed weak correlation (|r| < 0.25), indicating independence from one another. The one notable exception is screen_width and screen_height (r = 0.59), which is logically explained by standard device screen proportions. Features were left unchanged; the final decision on their use will be made based on model type during training.

### Data Transformation

#### Train/Test Split

In [ ]:
df_model = df_model.copy()
drop_cols = ["session_id", "client_id", "visit_date"]
df_model = df_model.drop(columns=drop_cols)
df_model.columns.tolist()

In [ ]:
X = df_model.drop(['target'], axis=1)
Y = df_model['target']

In [ ]:
print('X:', X.shape) 
print('Y:', Y.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.3, random_state=42, stratify=Y)

In [ ]:
print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('y_train:', y_train.shape)
print('y_test:', y_test.shape)

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

In [ ]:
numeric_cols = X_train.select_dtypes(include=['int8', 'int32', 'float64']).columns
categorical_cols = X_train.select_dtypes(include=['category']).columns

In [ ]:
print('Numeric columns:', list(numeric_cols))
print('Categorical columns:', list(categorical_cols))

In [ ]:
print(f"Target=1 share in train: {y_train.mean():.4f}")
print(f"Target=1 share in test: {y_test.mean():.4f}")

In [ ]:
ohe = OneHotEncoder(sparse_output = False, handle_unknown='ignore')
X_train_ohe = ohe.fit_transform(X_train[categorical_cols])
X_test_ohe = ohe.transform(X_test[categorical_cols])
ohe_col_names = ohe.get_feature_names_out(categorical_cols)
X_train_ohe = pd.DataFrame(X_train_ohe, columns= ohe_col_names, index=X_train.index)
X_test_ohe = pd.DataFrame(X_test_ohe, columns= ohe_col_names, index=X_test.index)

In [ ]:
X_train = X_train.drop(columns=categorical_cols)
X_test = X_test.drop(columns=categorical_cols)

In [ ]:
X_train = pd.concat([X_train, X_train_ohe], axis=1)
X_test = pd.concat([X_test, X_test_ohe], axis=1)

In [ ]:
print('X_train final shape:', X_train.shape)
print('X_test final shape:', X_test.shape)
print('Check that train and test have the same columns:', (X_train.columns == X_test.columns).all())
print('Y_train final shape:', y_train.shape)
print('Y_test final shape:', y_test.shape)

print('Missing values in X_train:', X_train.isna().sum().sum())
print('Missing values in X_test:', X_test.isna().sum().sum())

print('Categorical columns in X_train:')
print(X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist())

## Modeling

At this stage, a baseline (to estimate the minimum expected quality) and several classification models will be trained to predict whether a user performs the target action on the site. Since the target variable is binary (0 - target action not performed, 1 - performed), this is a binary classification task.

ROC-AUC will be used as the quality metric — it works well for tasks with strong class imbalance (target visits make up only ~2.7% of the total)

In [ ]:
models = {'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)}

In [ ]:
baseline = DummyClassifier(strategy='most_frequent', random_state=42)
baseline.fit(X_train, y_train)

baseline_pred_proba = baseline.predict_proba(X_test)[:, 1]
baseline_auc = roc_auc_score(y_test, baseline_pred_proba)

print(f"Baseline ROC-AUC: {baseline_auc:.4f}")

### Baseline

In [ ]:
results = [{'model': 'Baseline (Dummy)', 'ROC-AUC': baseline_auc, 'training_time_sec': 0}]

for model_name, model in models.items():
    print(f"Training: {model_name}...")
    start_time = time.time()
    
    model.fit(X_train, y_train)
    
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_proba)
    
    elapsed = time.time() - start_time
    print(f"{model_name}: ROC-AUC = {auc:.4f}, training time = {elapsed:.1f} sec")
    
    results.append({'model': model_name, 'ROC-AUC': auc, 'training_time_sec': elapsed})

### Model Selection

Model comparison results

All three models showed ROC-AUC noticeably above the baseline (0.5000):\
Logistic Regression: ROC-AUC = 0.6678 (553.2 sec)\
Decision Tree: ROC-AUC = 0.6795 (48.5 sec)\
Random Forest: ROC-AUC = 0.6865 (192.9 sec)

Training Logistic Regression triggered a convergence warning (ConvergenceWarning) — the result remains valid for comparison purposes.

Decision Tree was chosen as the final model. Although Random Forest showed a slightly higher ROC-AUC (difference under 1 p.p.), Decision Tree was chosen for the following reasons:\
Shorter training time — the model trains significantly faster (48.5 sec vs 192.9 sec for Random Forest), reducing compute cost.\
Interpretability — the structure of a single decision tree can be visualized and read as a set of understandable rules, which is valuable for specialists working with marketing campaigns.\
Minor quality loss — the ROC-AUC difference is under one percentage point and doesn't justify using a more complex model.

In [ ]:
dt_models = {'DT_depth_5': DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=5),
    'DT_depth_10': DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10),
    'DT_depth_15': DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=15),
    'DT_depth_10_leaf_20': DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10, min_samples_leaf=20),
    'DT_depth_10_leaf_50': DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10, min_samples_leaf=50)}

In [ ]:
dt_tuning_results = []
for model_name, model in dt_models.items():
    dt_result = model.fit(X_train, y_train)
    y_pred_proba = dt_result.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_pred_proba)
    dt_tuning_results.append({'model': model_name, 'ROC-AUC': auc})
    print(f"{model_name}: ROC-AUC = {auc:.4f}")

dt_tuning_df = pd.DataFrame(dt_tuning_results)
dt_tuning_df.sort_values('ROC-AUC', ascending= False)


After hyperparameter tuning, the best model was DecisionTreeClassifier with parameters max_depth=10, min_samples_leaf=20.
This model showed the highest ROC-AUC among the tested options, which is why it was chosen for the final training.

### Final Model Training

In [ ]:
best_model = DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=10, min_samples_leaf=20)

best_model.fit(X_train, y_train)

y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

In [ ]:
y_train_proba = best_model.predict_proba(X_train)[:, 1]
y_test_proba = best_model.predict_proba(X_test)[:, 1]

train_auc = roc_auc_score(y_train, y_train_proba)
test_auc = roc_auc_score(y_test, y_test_proba)

print(f"Train ROC-AUC: {train_auc:.4f}")
print(f"Test ROC-AUC: {test_auc:.4f}")
print(f"ROC-AUC gap: {train_auc - test_auc:.4f}")

In [ ]:
feature_importance = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

print(feature_importance.head(20))

In [ ]:
top_features = feature_importance.head(15)

plt.figure(figsize=(10, 6))
plt.barh(top_features.index[::-1], top_features.values[::-1])
plt.xlabel("Feature importance")
plt.title("Top-15 important features (Decision Tree, max_depth=10)")
plt.tight_layout()
plt.show()

The most influential group of features are specific traffic sources/campaigns (utm_source, utm_campaign), together accounting for more than half of the model's importance. These values are anonymized in the dataset, so specific campaign recommendations aren't possible, but with open data this analysis would allow pinpointing exactly which ad campaigns are worth scaling.

Among the interpretable features, the most significant are:\
visit_number - number of user visits: repeat visits strongly correlate with conversion, indicating the value of retargeting returning users;\
visit_hour - hour of visit: conversion is higher during daytime;\
device_os_os_not_detected and in-app browser features (Android Webview, Safari in-app) - visits through apps' built-in browsers (e.g. from a social media link) behave differently than visits from full browsers;\
screen_width/screen_height - moderately significant, likely indirectly reflecting the user's device type and class.

## Results

### Metrics Analysis

In [ ]:
metrics_df = pd.DataFrame({'Dataset': ['Train', 'Test'], 'ROC-AUC': [train_auc, test_auc]})
metrics_df

On the training set the model showed ROC-AUC of 0.6921, and on the test set - 0.6796. The test set ROC-AUC is noticeably above the random level of 0.5, indicating the model can distinguish between the target and non-target classes. The gap between the training and test sets is 0.0125, which is a small difference.

### Overfitting / Underfitting Analysis

The gap between the training and test ROC-AUC is 0.0125. The small gap between the metric values indicates no pronounced overfitting. At the same time, the test ROC-AUC of 0.6796 indicates a moderate ability of the model to distinguish between the target and non-target classes.

### Saving the Model

In [ ]:
joblib.dump(best_model, 'best_decision_tree.pkl')

In [ ]:
loaded_model = joblib.load('best_decision_tree.pkl')

row_x = X_test.sample(20, random_state=42)
row_y = y_test.loc[row_x.index]

predictions = loaded_model.predict(row_x)
predictions_proba = loaded_model.predict_proba(row_x)[:, 1]

comparison = pd.DataFrame({'real_target': row_y.values, 'predicted_target': predictions, 'predicted_probability': predictions_proba.round(4)})
comparison

A spot check on a small sample illustrates model behavior but doesn't replace the main quality metric — ROC-AUC on the full test set (0.6796), which accounts for probability ranking rather than hard classification at a 0.5 threshold.

### Final Conclusion

This project built a binary classification model to predict the probability that a user performs a target action on the "SberAutopodpiska" website, based on visit data (utm tags, device, geolocation, visit time).

Data. The source data (ga_sessions, ga_hits) was merged by session_id, and the target variable was formed as the presence of at least one target event in the session. The dataset is highly imbalanced — target visits make up ~2.7% of the total.

Data preparation. Missing values were cleaned while preserving their semantic meaning (e.g. distinguishing "no data for desktop" from "anomaly for mobile device"), categorical feature cardinality was reduced (top-N + "other" grouping), and new features were created (day of week, visit hour, screen width/height) with their relationship to the target checked. Several hidden technical anomalies were found and fixed: screen resolution placeholders, bots with an unrealistic 1600x20000 resolution.

Modeling. Four models were compared: baseline (ROC-AUC 0.50), Logistic Regression (0.668), Decision Tree (0.680), and Random Forest (0.687). Decision Tree was chosen as the final model based on the combination of quality, speed, and interpretability, despite a slight lag behind Random Forest in ROC-AUC. After hyperparameter tuning (max_depth=10, min_samples_leaf=20), the final model showed ROC-AUC = 0.6796 on the test set — above the task's target benchmark (~0.65) and with no signs of significant overfitting (train/test ROC-AUC gap = 0.0125).

Feature importance. The largest contributions to the prediction come from specific traffic sources and campaigns, along with behavioral features (number of visits, hour of visit) and device technical characteristics (undetected OS, in-app browsers).

Note on pipeline architecture. For the exploratory analysis, categorical grouping and missing-value imputation were performed on the full dataset for clarity and easier visualization of distributions. In the final production pipeline built for the API, all transformations (top-N category grouping, median values, outlier clipping bounds) are computed exclusively on the training set and stored as part of the pipeline for correct application to new data.

Result. The model is ready to be used as the basis for a service 
predicting conversion probability from visit attributes.